## Power BI Export
- Purpose: Consolidate all EDA outputs into 4 clean tables for Power BI
- Input: EDA 01, 02, 03 processed CSV files
- Output: 4 tables saved to `../data/processed/powerbi/`

## Table Structure
| Table | Source | Description |
|-------|--------|-------------|
| `dim_date` | EDA01 | Date, CPI, CPI_MoM, Cancel Search |
| `fact_churn` | EDA01+02+03 | Churn rates by segment, price tier, type |
| `fact_substitution` | EDA03 | Netflix vs Ad-free search trends |
| `fact_simulation` | EDA03 | Price increase scenarios + revenue index |

In [50]:
import pandas as pd
import numpy as np
import os

# Output directory 
output_dir = '../data/processed/powerbi/'
os.makedirs(output_dir, exist_ok=True)
print(f'Output directory ready: {output_dir}')

Output directory ready: ../data/processed/powerbi/


### Table 1: dim_date
- Date dimension table
- Contains: CPI trend + Cancel search intent
- Used in: Page 1 Lag Chart

In [51]:
# Load CPI data
cpi = pd.read_csv('../data/references/CPIRECSL_consumerPriceIndex_2023_2026.csv')
cpi.columns = ['date', 'CPI']
cpi['date'] = pd.to_datetime(cpi['date'])

# Filter from 2024 onwards (align with search data)
cpi = cpi[cpi['date'] >= '2024-01-01'].copy()
cpi['CPI_MoM'] = cpi['CPI'].pct_change(fill_method=None) * 100

# Load cancel search data
search = pd.read_csv('../data/references/time_series_US_20240101-20260222-1958.csv')
search.columns = ['date', 'cancel_search']
search['date'] = pd.to_datetime(search['date'])

# Merge on date
dim_date = pd.merge(cpi, search, on='date', how='outer').sort_values('date')
dim_date = dim_date.round(4)

# Save
dim_date.to_csv(f'{output_dir}dim_date.csv', index=False)
print(f'dim_date: {dim_date.shape}')
print(dim_date.head(3))

dim_date: (26, 4)
        date      CPI  CPI_MoM  cancel_search
0 2024-01-01  137.798      NaN             35
1 2024-02-01  138.000   0.1466             37
2 2024-03-01  137.868  -0.0957             28


### Table 2: fact_churn
- Churn analysis unified table
- Combines: Segment churn (EDA01) + Family/Individual (EDA02) + Segment x Price Tier (EDA03)
- Used in: Page 1 Bar Chart, Page 3 Threshold

In [52]:
# Source 1: Segment churn (EDA01) 
seg_churn = pd.read_csv('../data/processed/powerbi/eda01_segment_churn.csv')
seg_churn.columns = ['Segment', 'Churn_Rate']
seg_churn['Segment_Type'] = 'All'
seg_churn['PriceTier'] = 'All'
seg_churn['Source'] = 'EDA01_Segment'

# Source 2: Family vs Individual (EDA02) 
seg_type = pd.read_csv('../data/processed/powerbi/eda_02_churn_survive_rate.csv')
seg_type['Segment'] = 'All'
seg_type['PriceTier'] = 'All'
seg_type['Source'] = 'EDA02_SegmentType'
seg_type = seg_type[['Segment', 'PriceTier', 'Segment_Type', 'Churn_Rate', 'Source']]

# Source 3: Segment x Price Tier (EDA03) 
seg_tier = pd.read_csv('../data/processed/powerbi/eda_03_churn_by_segment_tier.csv')
seg_tier['Source'] = 'EDA03_SegmentTier'
seg_tier = seg_tier[['Segment', 'PriceTier', 'Segment_Type', 'Churn_Rate', 'Source']]

# Combine all 
fact_churn = pd.concat([seg_churn[['Segment','PriceTier','Segment_Type','Churn_Rate','Source']], 
                        seg_type, 
                        seg_tier], ignore_index=True)
fact_churn['Churn_Rate'] = fact_churn['Churn_Rate'].round(4)

# Save
fact_churn.to_csv(f'{output_dir}fact_churn.csv', index=False)
print(f'fact_churn: {fact_churn.shape}')
print(fact_churn['Source'].value_counts())

fact_churn: (23, 5)
Source
EDA03_SegmentTier    18
EDA01_Segment         3
EDA02_SegmentType     2
Name: count, dtype: int64


### Table 3: fact_substitution
- Substitution threat timeseries
- Contains: Netflix alternative vs Ad-free YouTube search
- Used in: Page 1 Substitution Chart

In [53]:
# Load substitution data
fact_substitution = pd.read_csv('../data/processed/powerbi/eda_03_substitution.csv')
fact_substitution.columns = ['date', 'netflix_alternative', 'adfree_youtube']
fact_substitution['date'] = pd.to_datetime(fact_substitution['date'])

# Add policy flag
fact_substitution['post_policy'] = fact_substitution['date'] >= '2025-10-01'

# Save
fact_substitution.to_csv(f'{output_dir}fact_substitution.csv', index=False)
print(f'fact_substitution: {fact_substitution.shape}')
print(fact_substitution.tail(3))

fact_substitution: (24, 4)
         date  netflix_alternative  adfree_youtube  post_policy
21 2025-12-01                   48              68         True
22 2026-01-01                   50              59         True
23 2026-02-01                   35              47         True


### Table 4: fact_simulation
- Price increase scenario simulation
- Contains: Revenue index + Churn by scenario + Price threshold
- Used in: Page 2 Simulator + Page 2 Threshold Chart

In [54]:
# # Load simulation results
# simulation = pd.read_csv('../data/processed/powerbi/eda_03_simulation_results.csv')

# #ADD BASELINE ROW (Option 2: Pre-Inflection)
# # Baseline represents 5+ month tenure users (high-LTV segment)
# # Family Churn: 16.31% (Pre-Inflection segment)
# # Individual Churn: 24.49% (Pre-Inflection segment)
# # 
# # Rationale:
# # - Eliminates noise from early churners (Fresh, Inflection segments)
# # - Explicitly incorporates H3 substitution threat (+197%)
# # - Provides conservative forecast for strategic decision-making
# # - Protects against overestimating revenue uplift

# baseline_row = pd.DataFrame({
#     'Price Increase': ['0%'],
#     'Family Price': [22.99],           # base_family from EDA 03
#     'Ind Price': [13.99],              # base_individual from EDA 03
#     'Family Churn': [16.31],           # Pre-Inflection Family Churn
#     'Ind Churn': [24.49],              # Pre-Inflection Individual Churn
#     'Family Rev Index': [0.0],         # Reference point (no revenue change)
#     'Individual Rev Index': [0.0],
#     'Ind Risk Index': [0.0],           # Reference point
#     'Total Rev Index': [0.0],           # Reference point
#     'Churn Increase': [0.0],
#     'Churn Penalty': [0.0],
#     'Performance Score': [0.0],
#     'Ind Risk Premium': [0.0]
# })

# # Prepend baseline to simulation results
# simulation = pd.concat([baseline_row, simulation], ignore_index=True)

# # Load price threshold
# threshold = pd.read_csv('../data/processed/powerbi/eda_03_price_threshold.csv')
# threshold.columns = ['PriceTierFine', 'Threshold_Churn_Rate', 'CI_Low', 'CI_High']

# simulation = simulation[['Price Increase', 'Family Price', 'Ind Price',
#                           'Family Churn', 'Ind Churn', 'Family Rev Index',
#                           'Ind Risk Index', 'Total Rev Index',
#                           'Churn Penalty', 'Performance Score']]

# # Save separately (different shapes, can't merge)
# simulation.to_csv(f'{output_dir}fact_simulation.csv', index=False)
# threshold.to_csv(f'{output_dir}fact_threshold.csv', index=False)

# print(f'fact_simulation: {simulation.shape}')
# print(f'fact_threshold: {threshold.shape}')
# print('\n*** fact_simulation (with Baseline) ***')
# print(simulation)

# # Load price threshold
# threshold = pd.read_csv('../data/processed/powerbi/eda_03_price_threshold.csv')
# threshold.columns = ['PriceTierFine', 'Threshold_Churn_Rate', 'CI_Low', 'CI_High']

# # Save separately (different shapes, can't merge)
# simulation.to_csv(f'{output_dir}fact_simulation.csv', index=False)
# threshold.to_csv(f'{output_dir}fact_threshold.csv', index=False)

# print(f'fact_simulation: {simulation.shape}')
# print(f'fact_threshold: {threshold.shape}')
# print(simulation)

### Table 5: fact_scenario_comparison

In [55]:
# scenario_comp = pd.read_csv(
#     '../data/processed/powerbi/eda_03_scenario_comparison.csv'
# )

# scenario_comp.to_csv(
#     f'{output_dir}fact_scenario_comparison.csv',
#     index=False
# )

### Export Summary

In [56]:
# print('Power BI Export Complete!')
# tables = {
# 'dim_date':          'date + CPI + search cancel  →  Page 1 Lag Chart',
# 'fact_churn':        'churn integration           →  Page 1 Bar + Page 3',
# 'fact_substitution': 'substitution threat         →  Page 1 Substitution',
# 'fact_simulation':   'scenario simulation + baseline        →  Page 2 Simulator',
# 'fact_threshold':    'price threshold             →  Page 2 Threshold',
# 'fact_scenario_comparison': 'decision support table      →  Page 2 Decision Support'
# }
# for table, desc in tables.items():
#     print(f'✔️ {table:<27}  {desc}')

In [57]:
# df = pd.read_csv('../data/processed/powerbi/eda_03_simulation_results.csv')
# print(df.columns.tolist())
# print(df.head())

In [59]:
# Load simulation results
simulation = pd.read_csv('../data/processed/powerbi/eda_03_simulation_results.csv')

# Baseline row
baseline_row = pd.DataFrame({
    'Price Increase': ['0%'],
    'Family Price': [22.99],
    'Ind Price': [13.99],
    'Family Churn': [16.31],
    'Ind Churn': [24.49],
    'Family Rev Index': [0.0],
    'Individual Rev Index': [0.0],
    'Ind Risk Index': [0.0],
    'Total Rev Index': [0.0],
    'Churn Increase': [0.0],
    'Churn Penalty': [0.0],
    'Performance Score': [0.0],
    'Ind Risk Premium': [0.0]
})

# Prepend baseline
simulation = pd.concat([baseline_row, simulation], ignore_index=True)

# Verify columns
print("Columns:", simulation.columns.tolist())
print(simulation[['Price Increase', 'Performance Score', 'Churn Penalty']])

# Save
simulation.to_csv(f'{output_dir}fact_simulation.csv', index=False)
# Save as new file for PowerBI
simulation.to_csv(f'{output_dir}fact_simulation_v2.csv', index=False)

# Load price threshold
threshold = pd.read_csv('../data/processed/powerbi/eda_03_price_threshold.csv')
threshold.columns = ['PriceTierFine', 'Threshold_Churn_Rate', 'CI_Low', 'CI_High']
threshold.to_csv(f'{output_dir}fact_threshold.csv', index=False)

print(f'fact_simulation: {simulation.shape}')
print(f'fact_threshold: {threshold.shape}')

Columns: ['Price Increase', 'Family Price', 'Ind Price', 'Family Churn', 'Ind Churn', 'Family Rev Index', 'Individual Rev Index', 'Ind Risk Index', 'Total Rev Index', 'Churn Increase', 'Churn Penalty', 'Performance Score', 'Ind Risk Premium']
  Price Increase  Performance Score  Churn Penalty
0             0%             0.0000         0.0000
1            +5%            16.2544         0.1603
2           +10%            16.7231         0.3207
3           +17%            16.8499         0.8122
4           +22%            16.8082         1.2239
5           +28%            16.6231         1.7776
fact_simulation: (6, 13)
fact_threshold: (5, 4)
